# Confusion Matrix Analysis
Compares per-class and pairwise confusion between the constant-depth baseline
and the stochastic-depth model on the CIFAR-10 test set.

**No GPU needed** — CPU runtime is fine, inference on 10k images is fast.

Run cells top to bottom. All figures and `summary.txt` are saved to Drive.

---
## 1. Mount Drive & copy code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
ROOT = '/content/drive/MyDrive/stoch_depth'

for fname in ['model.py', 'data.py', 'confusion_analysis.py']:
    src = f'{ROOT}/code/{fname}'
    dst = f'/content/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied {fname}')
    else:
        print(f'WARNING: {fname} not found at {src}')

---
## 2. Install dependencies

In [ ]:
!pip install scikit-learn --quiet
import sklearn
print('scikit-learn', sklearn.__version__)

---
## 3. Download CIFAR-10 test set
Skip this cell if `/content/data/cifar-10-batches-py` already exists.

In [ ]:
import torchvision
torchvision.datasets.CIFAR10(root='/content/data', train=False, download=True)
print('CIFAR-10 test set ready.')

---
## 4. Verify checkpoint paths
Edit the two paths below if yours differ.

In [ ]:
import os
ROOT = '/content/drive/MyDrive/stoch_depth'

CKPT_BASE  = f'{ROOT}/fig3/baseline/checkpoints/cifar10_pL1.0_modeconstant_n18_best.pt'
CKPT_STOCH = f'{ROOT}/fig3/stochastic/checkpoints/cifar10_pL0.5_modelinear_n18_best.pt'
OUT_DIR    = f'{ROOT}/figures/confusion'

for label, path in [('Baseline checkpoint ', CKPT_BASE),
                     ('Stochastic checkpoint', CKPT_STOCH)]:
    status = 'OK  ' if os.path.exists(path) else 'MISSING'
    print(f'{status}  {label}: {path}')

os.makedirs(OUT_DIR, exist_ok=True)
print(f'\nOutput dir: {OUT_DIR}')

---
## 5. Run analysis
Produces four PNG figures and `summary.txt` in the output dir.

In [ ]:
import os
os.environ['CKPT_BASE']  = CKPT_BASE
os.environ['CKPT_STOCH'] = CKPT_STOCH
os.environ['OUT_DIR']    = OUT_DIR

!python /content/confusion_analysis.py \
    --baseline   "$CKPT_BASE" \
    --stochastic "$CKPT_STOCH" \
    --data_root  /content/data \
    --out_dir    "$OUT_DIR"

---
## 6. Display results inline

In [ ]:
# Print summary table
ROOT = '/content/drive/MyDrive/stoch_depth'
OUT_DIR = f'{ROOT}/figures/confusion'

with open(f'{OUT_DIR}/summary.txt') as f:
    print(f.read())

In [ ]:
# Display all four figures
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT = '/content/drive/MyDrive/stoch_depth'
OUT_DIR = f'{ROOT}/figures/confusion'

figures = [
    ('confusion_baseline.png',   'Constant depth — confusion matrix'),
    ('confusion_stochastic.png', 'Stochastic depth — confusion matrix'),
    ('confusion_diff.png',       'Difference heatmap (stochastic − baseline)'),
    ('per_class_error.png',      'Per-class error rate'),
]

for fname, title in figures:
    path = f'{OUT_DIR}/{fname}'
    if not os.path.exists(path):
        print(f'Missing: {path}')
        continue
    img = mpimg.imread(path)
    fig, ax = plt.subplots(figsize=(9, 7) if 'per_class' not in fname else (10, 4))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 7. Quick reference — what to look for

Once the cells above have run, check `summary.txt` for:

| What | Where | Good sign |
|---|---|---|
| Overall accuracy | Top of summary | Stochastic higher |
| Per-class error Δ | Per-class table | Negative Δ = stochastic wins |
| `cat → dog` confusion rate | Pairwise table | Negative Δ confirms hypothesis |
| `automobile → truck` confusion rate | Pairwise table | Negative Δ confirms hypothesis |
| `confusion_diff.png` | Blue cells = stochastic better | Blue dominant off-diagonal |

**Share the summary.txt output and I will update the report with the actual numbers.**